In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
import gc

In [4]:
gc.collect()
df1 = pd.read_csv("./df_knni_rounded.csv", sep='\t').copy().reset_index(drop = True)
df1

,manufacturing_year,battery_capacity_kwh,odometer_km,vehicle_age_years,cycle_count,battery_health_percent,state_of_charge,depth_of_discharge,state_of_health,cell_voltage_avg,...,predicted_remaining_life_cycles,battery_failure,Vehicle_Brand_Num_Value,Vehicle_Model_Num_Value,Battery_Manufacturer_Num_Value,Terrain_Type_Num_Value,Battery_Chemistry_Num_Value,Drive_Type_Num_Value,Fleet_Or_Private_Num_Value,Vehicle_Type_Num_Value
0,2019.0,82.54,98163.0,6.94,250.0,73.61,85.230,26.34,72.93,3.60246,...,401.0,0.0,12.0,24.0,1.0,2.0,1.0,2.0,1.0,2.0
1,2018.0,64.17,113600.0,7.79,256.0,85.77,49.120,44.83,86.71,3.36580,...,1094.0,0.0,0.0,49.0,8.0,3.0,4.0,0.0,1.0,3.0
2,2018.0,90.92,242253.0,7.85,556.0,72.37,40.086,33.65,72.17,3.41520,...,645.0,0.0,17.0,48.0,3.0,2.0,4.0,2.0,0.0,6.0
3,2022.0,47.35,83700.0,6.44,353.0,85.11,54.320,37.73,82.88,3.42020,...,867.0,0.0,18.0,18.0,1.0,3.0,4.0,0.0,1.0,1.0
4,2015.0,68.04,88784.0,11.13,287.0,87.40,62.730,65.29,86.17,3.45560,...,1090.0,0.0,16.0,29.0,0.0,0.0,4.0,1.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,2016.0,45.99,182311.0,10.11,817.0,86.91,30.510,42.42,84.85,2.89010,...,2691.0,0.0,2.0,40.0,1.0,2.0,0.0,1.0,1.0,1.0
199996,2015.0,67.65,275317.0,11.06,840.0,65.08,77.610,30.76,65.08,3.57070,...,420.0,0.0,8.0,0.0,4.0,3.0,4.0,0.0,1.0,0.0
199997,2019.0,98.08,158747.0,4.89,336.0,86.08,44.590,24.64,84.90,2.98750,...,3148.0,0.0,2.0,40.0,0.0,4.0,0.0,1.0,0.0,2.0
199998,2016.0,51.49,95122.0,10.06,392.0,76.61,80.700,41.93,75.50,3.56980,...,797.0,0.0,16.0,30.0,7.0,0.0,4.0,1.0,1.0,1.0


In [12]:
class QDA_Class(object):
    def __init__(self, dataset, solver: str, shrinkage : str or float, priors, reg_param : float, 
                 store_covariance : bool, tol : float, covariance_estimator):
        self.dataset = dataset.copy().reset_index(drop = True)
        self.solver = solver
        self.shrinkage = shrinkage
        self.priors = priors
        self.reg_param = reg_param
        self.store_covariance = store_covariance
        self.tol = tol
        self.covariance_estimator = covariance_estimator
        
    def train_split(self):

        global X_train_global
        global X_test_global
        global y_train_global
        global y_test_global

        from sklearn.model_selection import train_test_split
        df4 = self.dataset.copy().reset_index(drop = True)
        Y_col = 'Vehicle_Model_Num_Value'
        X_cols = df4.loc[:, df4.columns != Y_col].columns
        # Split into 80% training and 20% testing
        X_train, X_test, y_train, y_test = train_test_split(
            df4[X_cols], df4[Y_col], test_size=0.2, random_state=42)
        X_train_global = X_train
        X_test_global = X_test
        y_train_global = y_train
        y_test_global = y_test

    def ML_fitting(self):
        global qda_fit_global

        qda = QuadraticDiscriminantAnalysis(solver = self.solver, shrinkage = self.shrinkage,
                                            priors = self.priors, reg_param = self.reg_param, 
                                            store_covariance = self.store_covariance, tol = self.tol, #
                                            covariance_estimator = self.covariance_estimator)
        qda_fit = qda.fit(X_train_global, y_train_global)
        qda_fit_global = qda_fit

    def ML_prediction(self):
        global qda_y_predict_global

        gc.collect()
        qda_y_predict = qda_fit_global.predict(X_test_global)
        qda_y_predict_global = qda_y_predict

    def evaluation(self):

        from sklearn.metrics import accuracy_score
        accuracy = accuracy_score(y_test_global, qda_y_predict_global)
        print("Accuracy:", accuracy)

        from sklearn.metrics import balanced_accuracy_score
        balanced_accuracy_score = balanced_accuracy_score(y_test_global, qda_y_predict_global)
        print("balanced_accuracy_score:", balanced_accuracy_score)

        from sklearn.metrics import top_k_accuracy_score
        top_k_accuracy_score = top_k_accuracy_score(y_test_global,qda_fit_global.predict_proba(X_test_global), k = 1)
        print("top_k_accuracy_score:", top_k_accuracy_score)

        from sklearn.metrics import average_precision_score
        average_precision_score = average_precision_score(y_test_global,qda_fit_global.predict_proba(X_test_global))
        print("average_precision_score:", average_precision_score)

        from sklearn.metrics import brier_score_loss
        brier_score_loss = brier_score_loss(y_test_global,qda_fit_global.predict_proba(X_test_global))
        print("brier_score_loss:", brier_score_loss)

        from sklearn.metrics import f1_score
        f1_score = f1_score(y_test_global, qda_y_predict_global, average='macro')
        print("f1_score_macro:", f1_score)
        from sklearn.metrics import f1_score
        f1_score_1 = f1_score(y_test_global, qda_y_predict_global, average='micro')
        print("f1_score_micro:", f1_score_1)
        f1_score_2 = f1_score(y_test_global, qda_y_predict_global, average='weighted')
        print("f1_score_weighted:", f1_score_2)
        f1_score_3 = f1_score(y_test_global, qda_y_predict_global, average=None)
        print("f1_score_None:", f1_score_3)

        from sklearn.metrics import log_loss
        log_loss = log_loss(y_test_global,qda_fit_global.predict_proba(X_test_global))
        print("log_loss:", log_loss)

        from sklearn.metrics import precision_score
        precision_score_macro = precision_score(y_test_global, qda_y_predict_global, average='macro')
        print("precision_score_macro:", precision_score_macro)
        precision_score_micro = precision_score(y_test_global, qda_y_predict_global, average='micro')
        print("precision_score_micro:", precision_score_micro)
        precision_score_weighted = precision_score(y_test_global, qda_y_predict_global, average='weighted')
        print("precision_score_weighted:", precision_score_weighted)
        precision_score_None = precision_score(y_test_global, qda_y_predict_global, average=None)
        print("precision_score_None:", precision_score_None)
        precision_score_None_ZD_1 = precision_score(y_test_global, qda_y_predict_global, average=None, zero_division=1)
        print("precision_score_None_ZD_1:", precision_score_None_ZD_1)
        precision_score_None_ZD_NAN = precision_score(y_test_global, qda_y_predict_global, average=None, zero_division=np.nan)
        print("precision_score_None_ZD_NAN:", precision_score_None_ZD_NAN)

        from sklearn.metrics import recall_score
        recall_score_macro = recall_score(y_test_global, qda_y_predict_global, average='macro')
        print("recall_score_macro:", recall_score_macro)
        recall_score_micro = recall_score(y_test_global, qda_y_predict_global, average='micro')
        print("recall_score_micro:", recall_score_micro)
        recall_score_weighted = recall_score(y_test_global, qda_y_predict_global, average='weighted')
        print("recall_score_weighted:", recall_score_weighted)
        recall_score_None = recall_score(y_test_global, qda_y_predict_global, average=None)
        print("recall_score_None:", recall_score_None)
        recall_score_None_ZD_1 = recall_score(y_test_global, qda_y_predict_global, average=None, zero_division=1)
        print("recall_score_None_ZD_1:", recall_score_None_ZD_1)
        recall_score_None_ZD_NAN = recall_score(y_test_global, qda_y_predict_global, average=None, zero_division=np.nan)
        print("recall_score_None_ZD_NAN:", recall_score_None_ZD_NAN)

        from sklearn.metrics import jaccard_score
        jaccard_score_macro = jaccard_score(y_test_global, qda_y_predict_global, average='macro')
        print("jaccard_score_macro:", jaccard_score_macro)
        jaccard_score_micro = jaccard_score(y_test_global, qda_y_predict_global, average='micro')
        print("jaccard_score_micro:", jaccard_score_micro)
        jaccard_score_weighted = jaccard_score(y_test_global, qda_y_predict_global, average='weighted')
        print("jaccard_score_weighted:", jaccard_score_weighted)
        jaccard_score_None = jaccard_score(y_test_global, qda_y_predict_global, average=None)
        print("jaccard_score_None:", jaccard_score_None)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovr = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), multi_class = 'ovr')
        print('roc_auc_score_ovr:', roc_auc_score_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_ovo = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), multi_class = 'ovo')
        print('roc_auc_score_ovo:', roc_auc_score_ovo)

        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovr = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovr')
        print("roc_auc_score_macro_ovr:", roc_auc_score_macro_ovr)
        from sklearn.metrics import roc_auc_score
        roc_auc_score_macro_ovo = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), average='macro',
                                           multi_class = 'ovo')
        print("roc_auc_score_macro_ovo:", roc_auc_score_macro_ovo)
        
        roc_auc_score_weighted_ovr = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovr')
        print("roc_auc_score_weighted_ovr:", roc_auc_score_weighted_ovr)
        roc_auc_score_weighted_ovo = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), 
                                               average='weighted', multi_class = 'ovo')
        print("roc_auc_score_weighted_ovo:", roc_auc_score_weighted_ovo)
        
        roc_auc_score_None_ovr = roc_auc_score(y_test_global, qda_fit_global.predict_proba(X_test_global), average=None,
                                              multi_class = 'ovr')
        print("roc_auc_score_None_ovr:", roc_auc_score_None_ovr)

        from sklearn.metrics import d2_log_loss_score
        d2_log_loss_score = d2_log_loss_score(y_test_global, qda_fit_global.predict_proba(X_test_global),
                                              sample_weight=None, labels=None)
        print("d2_log_loss_score:", d2_log_loss_score)

        from sklearn.metrics import d2_brier_score
        d2_brier_score = d2_brier_score(y_test_global, qda_fit_global.predict_proba(X_test_global), 
                                        sample_weight=None, labels=None)
        print("d2_brier_score:", d2_brier_score)
        

    def main(self):
        gc.collect()
        self.train_split()
        self.ML_fitting()
        self.ML_prediction()
        self.evaluation()
        gc.collect()
        

In [13]:
qda_1 = QDA_Class(df1, 'svd', None , None, 0.5, False, 1.0e-4, None)
qda_1.main()

Accuracy: 0.147875
balanced_accuracy_score: 0.1563872493357561
top_k_accuracy_score: 0.147875
average_precision_score: 0.12753332077524973
brier_score_loss: 1.062585305578055
f1_score_macro: 0.1382952003999891
f1_score_micro: 0.147875
f1_score_weighted: 0.13600712402801693
f1_score_None: [0.29761905 0.32750243 0.1930942  0.15044248 0.05211726 0.0781759
 0.21318681 0.16639118 0.21731343 0.19458763 0.18385386 0.16501146
 0.08406305 0.10149254 0.07013575 0.06294865 0.13273341 0.06870706
 0.12953368 0.13001524 0.13765542 0.11086798 0.09859155 0.12710765
 0.09060176 0.06896552 0.05145798 0.16264522 0.17923309 0.17142857
 0.16817965 0.03608847 0.12473573 0.0973913  0.08545035 0.06593407
 0.04301075 0.20849421 0.14829268 0.03087886 0.11297071 0.1221374
 0.13024602 0.10609481 0.14344828 0.18859649 0.30945559 0.182881
 0.17665289 0.15197568 0.18229167 0.2051901  0.09141274 0.14305115
 0.13389562]
log_loss: 4.5174607900799275
precision_score_macro: 0.14882709843812203
precision_score_micro: 0.14

In [14]:
qda_fit_global

,"reg_param reg_param: float, default=0.0Regularizes the per-class covariance estimates by transforming S2 as``S2 = (1 - reg_param) * S2 + reg_param * np.eye(n_features)``,where S2 corresponds to the `scaling_` attribute of a given class.",0.5
,"solver solver: {'svd', 'eigen'}, default='svd'Solver to use, possible values: - 'svd': Singular value decomposition (default). Does not compute the covariance matrix, therefore this solver is recommended for data with a large number of features. - 'eigen': Eigenvalue decomposition. Can be combined with shrinkage or custom covariance estimator.",'svd'
,"shrinkage shrinkage: 'auto' or float, default=NoneShrinkage parameter, possible values: - None: no shrinkage (default). - 'auto': automatic shrinkage using the Ledoit-Wolf lemma. - float between 0 and 1: fixed shrinkage parameter. Enabling shrinkage is expected to improve the model when some classes have a relatively small number of training data points compared to the number of features by mitigating overfitting during the covariance estimation step.This should be left to `None` if `covariance_estimator` is used.Note that shrinkage works only with 'eigen' solver.",None
,"priors priors: array-like of shape (n_classes,), default=NoneClass priors. By default, the class proportions are inferred from thetraining data.",None
,"store_covariance store_covariance: bool, default=FalseIf True, the class covariance matrices are explicitly computed andstored in the `self.covariance_` attribute... versionadded:: 0.17",False
,"tol tol: float, default=1.0e-4Absolute threshold for the covariance matrix to be considered rankdeficient after applying some regularization (see `reg_param`) to each`Sk` where `Sk` represents covariance matrix for k-th class. Thisparameter does not affect the predictions. It controls when a warningis raised if the covariance matrix is not full rank... versionadded:: 0.17",0.0001
,"covariance_estimator covariance_estimator: covariance estimator, default=NoneIf not None, `covariance_estimator` is used to estimate the covariancematrices instead of relying on the empirical covariance estimator(with potential shrinkage). The object should have a fit method anda ``covariance_`` attribute like the estimators in:mod:`sklearn.covariance`. If None the shrinkage parameter drives theestimate.This should be left to `None` if `shrinkage` is used.Note that `covariance_estimator` works only with the 'eigen' solver.",None
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)Unique class labels.","ndarray[float64](55,)","[ 0., 1., 2.,...,52.,53.,54.]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](67,)","['manufacturing_year','battery_capacity_kwh','odometer_km',..., 'Drive_Type_Num_Value','Fleet_Or_Private_Num_Value', 'Vehicle_Type_Num_Value']"
"means_ means_: array-like of shape (n_classes, n_features)Class-wise means.","ndarray[float64](55, 67)","[[ 2019.54, 76.45,103545.13,..., 0.93, 0.73, 2.17], [ 2019.54, 76.86,101464.53,..., 0.94, 0.72, 2.19], [ 2019.44, 76.38,105787.89,..., 0.97, 0.71, 2.23], ..., [ 2019.58, 76.26,101821.49,..., 0.96, 0.71, 2.16], [ 2019.7 , 76.25, 97850.15,..., 0.94, 0.74, 2.23], [ 2019.46, 75.95,101829.88,..., 0.97, 0.73, 2.2 ]]"
